# Notebook 4 - the CV Validation with Cleanlab Library 

Independent check: does confident-learning flag the same PLR as the hand-built watchlist?


In [1]:
#load libraries
import numpy as np
import pandas as pd
from cleanlab.filter import find_label_issues

/Users/lauragies/Documents/Job/NF_Bootcamp/capstone-kiezkeeper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#XGBoost OOF probabilities (527 PLR) — the basis 
oof = pd.read_csv("../../models/M_oof_xgboost.csv", dtype={"plr_id": str})
oof["plr_id"] = oof["plr_id"].astype(str).str.zfill(8)

# the resemblance list (25 PLR, pure oof_prob) — this is what Cleanlab validates
resemblance = pd.read_csv("../../models/M2_resemblance_list.csv", dtype={"plr_id": str})
resemblance["plr_id"] = resemblance["plr_id"].astype(str).str.zfill(8)

print(f"{len(oof)} PLR total | {int(oof['y_true'].sum())} designated | "
      f"{len(resemblance)} on resemblance list")

527 PLR total | 109 designated | 25 on resemblance list


In [3]:
#Cleanlab needs given labels + out-of-sample pred_probs as an (n, n_classes) matrix
#Class 0 = undesignated, Class 1 = designated; our oof_prob is P(class 1)
labels = oof["y_true"].to_numpy().astype(int)
pred_probs = np.column_stack([1 - oof["oof_prob"], oof["oof_prob"]])  # shape (527, 2)

#find_label_issues: PLR whose given label is inconsistent with the model's confident prediction
#ranked by likelihood of being an issue (most suspicious first)
issue_idx = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence",
)

#assemble the flagged PLR with their info
cl_issues = oof.iloc[issue_idx].copy()
cl_issues["cl_rank"] = np.arange(1, len(cl_issues) + 1)

#we care about issues among UNDESIGNATED PLR (label 0 flagged as looking designated)
#the confident-learning analog of our early-warning watchlist
cl_undes = cl_issues[cl_issues["y_true"] == 0].copy()

print(f"Cleanlab flagged {len(cl_issues)} label issues total")
print(f"  of which {len(cl_undes)} are undesignated PLR (early-warning analog)")
print(f"  and {len(cl_issues) - len(cl_undes)} are designated PLR flagged as not-designated-looking")

Cleanlab flagged 64 label issues total
  of which 38 are undesignated PLR (early-warning analog)
  and 26 are designated PLR flagged as not-designated-looking


In [4]:
#Overlap: how many of the 25 resemblance PLR does Cleanlab independently flag?
rl_ids = set(resemblance["plr_id"])
cl_undes_ids = set(cl_undes["plr_id"])

overlap        = rl_ids & cl_undes_ids
only_resemb    = rl_ids - cl_undes_ids
only_cleanlab  = cl_undes_ids - rl_ids

print(f"Resemblance list:            {len(rl_ids)} PLR")
print(f"Cleanlab undesignated flags: {len(cl_undes_ids)} PLR")
print(f"Overlap (both agree):        {len(overlap)} PLR "
      f"({len(overlap)/len(rl_ids):.0%} of the resemblance list)")
print(f"Only on resemblance list:    {len(only_resemb)}")
print(f"Only flagged by Cleanlab:    {len(only_cleanlab)}")

if only_resemb:
    print("\nOn resemblance list but not flagged by Cleanlab:")
    print(resemblance[resemblance["plr_id"].isin(only_resemb)]
          [["plr_name", "bez", "oof_prob"]].to_string(index=False))

Resemblance list:            25 PLR
Cleanlab undesignated flags: 38 PLR
Overlap (both agree):        25 PLR (100% of the resemblance list)
Only on resemblance list:    0
Only flagged by Cleanlab:    13


In [5]:
# where do the 13 Cleanlab-only PLR rank on oof_prob? (verify they're near-misses, not scattered)
cl_only = oof[oof["plr_id"].isin(only_cleanlab)].copy()
cl_only["oof_rank"] = oof["oof_prob"].rank(ascending=False, method="min").astype(int)
cl_only = oof.assign(oof_rank=oof["oof_prob"].rank(ascending=False, method="min").astype(int))
print(cl_only[cl_only["plr_id"].isin(only_cleanlab)]
      .sort_values("oof_prob", ascending=False)[["plr_id","oof_prob","oof_rank"]].to_string(index=False))

  plr_id  oof_prob  oof_rank
05100316  0.766517        88
01100205  0.756666        90
04501042  0.712941        91
12100103  0.712730        92
09200716  0.704122        93
12100204  0.694774        95
01100415  0.693884        96
09100304  0.689616        98
08100101  0.658168        99
04501151  0.638546       100
04501149  0.616660       101
01200624  0.604541       103
04501043  0.586272       104


In [8]:
milieu = (pd.read_csv("../../data/Module 2/target_milieuschutz.csv", dtype={"plr_id": str})
            .assign(plr_id=lambda d: d["plr_id"].astype(str).str.zfill(8)))

cl13 = oof[oof["plr_id"].isin(only_cleanlab)].merge(
    milieu[["plr_id","milieu_anteil"]], on="plr_id", how="left")
print(cl13[["plr_id","oof_prob","milieu_anteil"]]
      .sort_values("oof_prob", ascending=False).to_string(index=False))

  plr_id  oof_prob  milieu_anteil
05100316  0.766517   1.801573e-02
01100205  0.756666   2.602949e-08
04501042  0.712941   5.816938e-09
12100103  0.712730   0.000000e+00
09200716  0.704122   0.000000e+00
12100204  0.694774   1.523684e-01
01100415  0.693884   0.000000e+00
09100304  0.689616   0.000000e+00
08100101  0.658168   1.332604e-01
04501151  0.638546   0.000000e+00
04501149  0.616660   0.000000e+00
01200624  0.604541   3.745172e-02
04501043  0.586272   5.227920e-04


In [9]:
meta = (pd.read_csv("../../data/final_datasets/dataset_0.csv", dtype={"plr_id": str})
          .assign(plr_id=lambda d: d["plr_id"].astype(str).str.zfill(8))[["plr_id","bez"]])
cl13 = cl13.merge(meta, on="plr_id", how="left")
cl13["bez_name"] = cl13["bez"].str.split(" - ").str[-1].str.strip()
cl13["weak_fold"] = cl13["bez_name"].isin(["Charlottenburg-Wilmersdorf","Lichtenberg"])
print(cl13[["plr_id","oof_prob","milieu_anteil","bez_name","weak_fold"]]
      .sort_values("oof_prob", ascending=False).to_string(index=False))
print(f"\n{cl13['weak_fold'].sum()} of {len(cl13)} Cleanlab-only PLR are in the weak fold")

  plr_id  oof_prob  milieu_anteil                   bez_name  weak_fold
05100316  0.766517   1.801573e-02                    Spandau      False
01100205  0.756666   2.602949e-08                      Mitte      False
04501042  0.712941   5.816938e-09 Charlottenburg-Wilmersdorf       True
12100103  0.712730   0.000000e+00              Reinickendorf      False
09200716  0.704122   0.000000e+00           Treptow-Köpenick      False
12100204  0.694774   1.523684e-01              Reinickendorf      False
01100415  0.693884   0.000000e+00                      Mitte      False
09100304  0.689616   0.000000e+00           Treptow-Köpenick      False
08100101  0.658168   1.332604e-01                   Neukölln      False
04501151  0.638546   0.000000e+00 Charlottenburg-Wilmersdorf       True
04501149  0.616660   0.000000e+00 Charlottenburg-Wilmersdorf       True
01200624  0.604541   3.745172e-02                      Mitte      False
04501043  0.586272   5.227920e-04 Charlottenburg-Wilmersdorf    

In [10]:
# all 38 Cleanlab undesignated flags with their oof_prob rank — is there really a gap 26-87?
cl_all = oof.assign(oof_rank=oof["oof_prob"].rank(ascending=False, method="min").astype(int))
cl_all = cl_all[cl_all["plr_id"].isin(cl_undes["plr_id"])]
print(cl_all.sort_values("oof_rank")[["plr_id","oof_prob","oof_rank"]].to_string(index=False))

  plr_id  oof_prob  oof_rank
01401044  0.999109         3
01300731  0.997601        13
01100416  0.995976        19
07100206  0.992505        23
01200623  0.986148        28
01401049  0.985959        29
04501147  0.984600        30
01300836  0.984006        32
01200522  0.983263        34
01200628  0.979387        35
09200511  0.977478        37
01300834  0.976187        40
02200211  0.975650        42
03601142  0.963523        47
07300619  0.916001        63
01100309  0.909953        64
09100101  0.895231        65
08100102  0.893927        66
01400937  0.891792        67
07200414  0.871418        70
04300414  0.871022        71
04300416  0.862824        72
01400938  0.838907        77
12100102  0.824959        79
04501150  0.823657        80
05100316  0.766517        88
01100205  0.756666        90
04501042  0.712941        91
12100103  0.712730        92
09200716  0.704122        93
12100204  0.694774        95
01100415  0.693884        96
09100304  0.689616        98
08100101  0.65

## bottomline

Cleanlab flags 38 undesignated areas continuously down the probability ranking; the top 25 of them are exactly your resemblance list (100% overlap), and the remaining 13 are the lower-ranked continuation past your N=25 cutoff. 

## Result

All 32 of the watchlist's planning areas are independently flagged by Cleanlab (100% overlap). Cleanlab, using a completely different method (label-quality/confident-learning on the training labels, not the oof_prob ranking) points at every single area that the watchlist surfaces. That's strong convergent validation: two independent signals agree that these 32 undesignated PLR look designation-like.

# notes 

- my list has a threshold, cleanlab doesnt
- cleanlab flags every PLR whose given label looks inconsistent with the model's confident prediction, by its own internal confident-learning criterion
- watchlist of 32 are all flagged by the cleanlab validation ! agrees with everything we flagged and would have flagged more. 

## what it could mean that cleanlab selected 436 additional planning areas from the weakest fold: 

It could mean the weak fold is under-flagged by your oof_prob ranking — the model's sparse-fold scores are depressed, those three CW PLR score 0.59–0.64 and miss the cutoff, but Cleanlab (working from label quality, not the fold-limited probabilities) still catches them. If so, the weak fold is costing you real candidates, and these three are evidence of it. That's a genuine finding worth a sentence.
Or it could mean Cleanlab is picking up the same label-sparsity noise in that district and flagging on thin evidence. Cleanlab's confident-learning is only as good as the label signal, and CW is where the label signal is weakest.

--> Cleanlab surfaces 6 further candidates just below the watchlist cutoff (oof_prob 0.59–0.69); notably three are in Charlottenburg-Wilmersdorf, the weak-labelling fold, consistent with that district's known score suppression.

The Charlottenburg-Wilmersdorf point stands and is arguably sharper now: three of the six are in the weak fold, scoring 0.586–0.639 — well below both your boundaries. Given the fold suppresses oof_prob there, it's plausible their "true" similarity is higher and the fold is what keeps them out. That's the finding worth stating: Cleanlab, not being bound by the fold-limited probabilities, flags them anyway.

### Result Discussion - umschreiben

Cleanlab, applied to the same XGBoost out-of-fold probabilities via confident learning, independently flags all 25 hand-built watchlist PLR as label issues — a 100% overlap. This confirms the watchlist is not an artefact of the chosen rank cutoff: a formal, established method identifies exactly the same core set.

Cleanlab additionally flags 13 further undesignated PLR, all with oof_prob between 0.59 and 0.77 — i.e. just below the N=25 threshold (0.824) — which is consistent with the deliberately conservative cutoff placed at the last visible gap in the probability ranking; below it the probabilities decline smoothly with no natural boundary. 

The two methods therefore agree completely on the core watchlist and differ only at the margin, where the choice of boundary is inherently a judgment call rather than a data-driven one. One caveat: because both approaches rest on the same out-of-fold probabilities, Cleanlab validates the selection rule, not the underlying probabilities — it is a strong consistency check, not a fully independent second source.

These six aren't "surfaced only by Cleanlab because the weak fold hid them." They're the immediate next tier on both the orderings: rank_oof 31–38, and for the zero-coverage CW ones, rank_urgency 26–31. Cleanlab, the oof ranking, and the urgency ranking all three agree they're the strongest candidates just past the line. The three methods draw the boundary within a handful of ranks of each other. 

This is perceived as as strong convergent-validation rather than "the fold hid candidates". The question is rather where to draw the line than what the models says.


### Exploration 

In [8]:
# --- feature profile of the watchlist vs. (a) other undesignated PLR, (b) designated ---
# RAW dataset_0 values (interpretable). Descriptive only — the watchlist was selected
# by oof_prob built from these features, so differences are expected, not causal.

feats = pd.read_csv("../../data/final_datasets/dataset_0.csv", dtype={"plr_id": str})
feats["plr_id"] = feats["plr_id"].astype(str).str.zfill(8)

# attach designation label + watchlist membership
feats = feats.merge(oof[["plr_id", "y_true"]], on="plr_id", how="left")
wl_ids = set(watchlist["plr_id"])
feats["group"] = np.where(
    feats["y_true"] == 1, "designated",
    np.where(feats["plr_id"].isin(wl_ids), "watchlist", "undes_rest")
)

# feature columns only: the three axes, by prefix. Excludes id/name/bez/target/oof.
feat_cols = [c for c in feats.columns if c.startswith(("soc_", "re_", "com_"))]
print(f"{len(feat_cols)} features | groups: {feats['group'].value_counts().to_dict()}\n")

# group means (raw, interpretable)
means = feats.groupby("group")[feat_cols].mean().T
means = means[["watchlist", "undes_rest", "designated"]]   # column order

# standardized differences (comparable across features): (meanA - meanB) / pooled SD
def std_diff(a, b):
    ma, mb = feats.loc[feats.group==a, feat_cols].mean(), feats.loc[feats.group==b, feat_cols].mean()
    sa, sb = feats.loc[feats.group==a, feat_cols].std(),  feats.loc[feats.group==b, feat_cols].std()
    pooled = np.sqrt((sa**2 + sb**2) / 2).replace(0, np.nan)
    return (ma - mb) / pooled

means["d_vs_rest"]      = std_diff("watchlist", "undes_rest")
means["d_vs_designated"] = std_diff("watchlist", "designated")

# what most separates the watchlist from the OTHER undesignated PLR
print("=== Watchlist vs. other undesignated — top separating features ===")
print(means.reindex(means["d_vs_rest"].abs().sort_values(ascending=False).index)
           [["watchlist", "undes_rest", "d_vs_rest"]].head(12).round(3).to_string())

# how the watchlist compares to the DESIGNATED areas it's meant to resemble
print("\n=== Watchlist vs. designated — where they still differ most ===")
print(means.reindex(means["d_vs_designated"].abs().sort_values(ascending=False).index)
           [["watchlist", "designated", "d_vs_designated"]].head(12).round(3).to_string())

42 features | groups: {'undes_rest': 386, 'designated': 109, 'watchlist': 32}

=== Watchlist vs. other undesignated — top separating features ===
group                                                         watchlist  undes_rest  d_vs_rest
re_altbau_share                                                   0.450       0.128      1.663
com_gastro_share_level_2026                                       0.058       0.033      1.386
soc_young_to_middle_adult_share_2025                              0.452       0.365      1.302
soc_average_household_size_2024                                   1.665       1.849     -1.221
re_brw_trend                                                     -0.357      -0.179     -1.179
com_n_gastro                                                     74.370      19.350      1.106
re_brw_niveau                                                  2803.723    1221.377      0.934
re_dichte_all                                                    12.976       4.259      0.911

the watchlist is much closer to designated than to undesignated areas (resemblance validated), the commercial dimension is a real discriminator (contribution validated), and the small directional gaps hint at "on the way, not arrived" — offered as a hypothesis, not a result. That's three clean sentences, each backed by a specific number, and honest about which are solid (the first two) and which are suggestive (the third).

In [9]:
# --- vulnerable-population read: is there still a displaceable population? -----
# LEVEL = how much vulnerable pop is still present (the actual question)
# CHANGE = whether displacement already appears to be underway
# Descriptive only; watchlist selected on oof_prob, so read as characterization.

vuln_cols = [c for c in feats.columns
             if any(k in c for k in ["transfer_benefit", "unemployment"])]
print("vulnerability-related columns found:", vuln_cols, "\n")

# group means for level + change side by side
prof = feats.groupby("group")[vuln_cols].mean().T
prof = prof[["watchlist", "undes_rest", "designated"]].round(4)
print(prof.to_string(), "\n")

# per-PLR view of the watchlist: level AND change together, so you can see
# which areas still have a population AND whether it's already declining
level_col  = [c for c in vuln_cols if "transfer_benefit" in c and "change" not in c]
change_col = [c for c in vuln_cols if "transfer_benefit" in c and "change" in c]

if level_col and change_col:
    lv, ch = level_col[0], change_col[0]
    wl_view = (feats[feats["group"] == "watchlist"]
               [["plr_id", lv, ch]]
               .merge(watchlist[["plr_id", "plr_name", "bez"]], on="plr_id", how="left")
               .sort_values(lv, ascending=False))
    print(f"Watchlist by transfer-benefit LEVEL ({lv}) with CHANGE ({ch}):")
    print(wl_view[["plr_name", "bez", lv, ch]].round(4).to_string(index=False))
else:
    print("Couldn't identify separate level/change columns — check names in:", vuln_cols)

vulnerability-related columns found: ['soc_unemployment_share_change_2020_2024', 'soc_transfer_benefit_share_2024', 'soc_transfer_benefit_share_change_2020_2024'] 

group                                        watchlist  undes_rest  designated
soc_unemployment_share_change_2020_2024        -0.0064     -0.0011     -0.0081
soc_transfer_benefit_share_2024                 0.1122      0.0918      0.1226
soc_transfer_benefit_share_change_2020_2024    -0.0198     -0.0069     -0.0204 

Watchlist by transfer-benefit LEVEL (soc_transfer_benefit_share_2024) with CHANGE (soc_transfer_benefit_share_change_2020_2024):
              plr_name                             bez  soc_transfer_benefit_share_2024  soc_transfer_benefit_share_change_2020_2024
           Schulstraße                      01 - Mitte                            0.242                                       -0.026
         Brunnenstraße                      01 - Mitte                            0.239                                   

Across the 32 watchlist areas, the mean transfer-benefit share in 2024 is 0.112, 
sitting almost exactly between the undesignated remainder (0.092) and the already-designated 
areas (0.123), which indicates that the flagged neighbourhoods still retain a vulnerable 
population comparable to those already under protection. At the same time, this share is 
declining in the watchlist areas at the same rate as in the designated ones 
(−0.020 over 2020–2024) and faster than in the undesignated remainder (−0.007), 
suggesting that displacement pressure is not merely anticipated but already underway. 

The per-area picture, however, is strongly bimodal and should not be reduced to the mean: 
the top group — led by five Mitte areas such as Koloniestraße (level 0.212, change −0.051) 
and Schulstraße (0.242, −0.026) — combines a large remaining vulnerable population with 
a steep decline, marking them as the areas where both the need and the urgency for protection
are highest. 

A second group, including Arkonaplatz (0.033), Falkplatz (0.036) and Invalidenstraße (0.037), 
resembles designated areas on the structural features the model relies on but retains 
little benefit-dependent population, implying that on this particular indicator the 
displacement of lower-income residents has largely already occurred. It should be stressed 
that transfer-benefit share captures only one, income-based dimension of vulnerability 
and cannot detect displaceable middle-income or long-lease tenants, so a low value does 
not establish that an area is no longer at risk. 

Taken together, the results support the early-warning framing for the higher-vulnerability 
subset while honestly flagging that a portion of the watchlist reflects structural resemblance
to designated areas more than a currently protectable population.
